In [1]:
!pip install genesis-world==0.3.10
!pip install h5py

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [2]:
"""
PHASE 2- Data Collection.
"""

import genesis as gs
import numpy as np
import math
import os
import h5py

# ---------------- CONFIG ----------------
sim_dt = 1.0 / 20.0          # was 1/30  -> faster
video_fps = 10

SAVE_DIR = "datasets/phase2"
OUT_H5 = os.path.join(SAVE_DIR, "combined_dataset_fast_strict.h5")

NUM_EPISODES = 80
EPISODE_DURATION = 15.0      # was 25 -> faster
STOP_ON_COLLISION = True
KEEP_ONLY_SUCCESS = True
MIN_CLEARANCE_FILTER = 0.0

SAVE_TOP_VIDEO = False       # keep False for max speed
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------- RESET / INIT ----------------
try:
    gs.destroy()
except Exception:
    pass
gs.init(backend=gs.cuda, logging_level="warning")

# ---------------- TRACK ----------------
OVAL_RADIUS = 15.0
STRAIGHT_LEN = 40.0
TRACK_WIDTH = 10.0
NUM_CONES = 60
perimeter = 2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS

CONE_R = 0.30
LANE_HALF = TRACK_WIDTH / 2.0
CAR_HALF_W = 1.25
LANE_SAFE = float(LANE_HALF - (CAR_HALF_W + CONE_R + 0.25))
print("LANE_SAFE =", LANE_SAFE)

def wrap_pi(a):
    return (a + math.pi) % (2 * math.pi) - math.pi

def get_oval_pos(dist, offset=0.0):
    d = dist % perimeter
    if d < STRAIGHT_LEN:
        return (OVAL_RADIUS + offset, -STRAIGHT_LEN / 2 + d)
    d -= STRAIGHT_LEN
    if d < math.pi * OVAL_RADIUS:
        ang = (d / (math.pi * OVAL_RADIUS)) * math.pi
        R = (OVAL_RADIUS + offset)
        return (R * math.cos(ang), STRAIGHT_LEN / 2 + R * math.sin(ang))
    d -= math.pi * OVAL_RADIUS
    if d < STRAIGHT_LEN:
        return (-OVAL_RADIUS - offset, STRAIGHT_LEN / 2 - d)
    d -= STRAIGHT_LEN
    ang = math.pi + (d / (math.pi * OVAL_RADIUS)) * math.pi
    R = (OVAL_RADIUS + offset)
    return (R * math.cos(ang), -STRAIGHT_LEN / 2 + R * math.sin(ang))

def get_tangent_normal(dist, eps=0.25):
    x1, y1 = get_oval_pos(dist, 0.0)
    x2, y2 = get_oval_pos(dist + eps, 0.0)
    tx, ty = (x2 - x1), (y2 - y1)
    nrm = math.sqrt(tx * tx + ty * ty) + 1e-8
    tx, ty = tx / nrm, ty / nrm
    nx, ny = -ty, tx
    return (tx, ty), (nx, ny)

def project_to_centerline(px, py, s_guess, window=16.0, samples=21):
    best_s = s_guess
    best_d2 = 1e30
    s0 = s_guess - window
    ds = (2 * window) / (samples - 1)
    for i in range(samples):
        s = s0 + i * ds
        cx, cy = get_oval_pos(s, 0.0)
        dx, dy = px - cx, py - cy
        d2 = dx * dx + dy * dy
        if d2 < best_d2:
            best_d2 = d2
            best_s = s
    best_s = best_s % perimeter
    cx, cy = get_oval_pos(best_s, 0.0)
    (tx, ty), (nx, ny) = get_tangent_normal(best_s)
    cte = (px - cx) * nx + (py - cy) * ny
    return best_s, cx, cy, tx, ty, nx, ny, float(cte)

# ---------------- SCENE (no renderer) ----------------
scene = gs.Scene(sim_options=gs.options.SimOptions(dt=sim_dt, substeps=1),
                 show_viewer=False, renderer=None)
scene.add_entity(gs.morphs.Plane())

# cones fixed
cone_xy = []
for i in range(NUM_CONES):
    dist = (i / NUM_CONES) * perimeter
    for side in (-1, +1):
        cx, cy = get_oval_pos(dist, offset=side * LANE_HALF)
        scene.add_entity(gs.morphs.Cylinder(radius=CONE_R, height=1.0, pos=(cx, cy, 0.5), fixed=True))
        cone_xy.append((float(cx), float(cy), float(CONE_R)))
cone_xy = np.array(cone_xy, dtype=np.float32)

# obstacles pool fixed
OB_SIZE = 1.5
MAX_OBS = 10
obstacle_entities = []
for _ in range(MAX_OBS):
    obstacle_entities.append(scene.add_entity(gs.morphs.Box(size=(OB_SIZE, OB_SIZE, OB_SIZE), pos=(0, 0, -100), fixed=True)))

# car
start_x, start_y = get_oval_pos(0.0, 0.0)
car = scene.add_entity(gs.morphs.URDF(file="assets/smart_car.urdf", fixed=True, pos=(start_x, start_y - 5.0, 0.8)))
scene.build()
scene.step()

# ---------------- HELPERS (unchanged logic) ----------------
def yaw_to_quat_wxyz(yaw):
    h = 0.5 * float(-yaw)  # your sign fix
    return (math.cos(h), 0.0, 0.0, math.sin(h))

def set_pose_yaw(ent, px, py, pz, yaw):
    ent.set_pos((float(px), float(py), float(pz)))
    ent.set_quat(yaw_to_quat_wxyz(yaw))

def set_joint_positions(vals):
    if not hasattr(car, "set_dofs_position"):
        return
    car.set_dofs_position(np.array(vals, dtype=np.float32),
                          dofs_idx_local=np.array([0, 1], dtype=np.int32))

# lidar
N_RAYS = 41
FOV = math.radians(170.0)
ray_angles = np.linspace(-FOV / 2, FOV / 2, N_RAYS).astype(np.float32)
LIDAR_MAX = 30.0

def lidar_scan_fast(px, py, yaw, ray_angles, obs_boxes, max_range=30.0):
    if len(obs_boxes) == 0:
        return np.full(len(ray_angles), max_range, dtype=np.float32)
    ga = ray_angles + yaw
    dx = np.sin(ga).astype(np.float32)
    dy = np.cos(ga).astype(np.float32)
    P = np.array([px, py], dtype=np.float32).reshape(1, 1, 2)
    D = np.stack([dx, dy], axis=1).reshape(-1, 1, 2)
    B = obs_boxes.reshape(1, -1, 4)
    Bmin = B[..., :2] - B[..., 2:]
    Bmax = B[..., :2] + B[..., 2:]
    invD = 1.0 / (D + 1e-9)
    t0 = (Bmin - P) * invD
    t1 = (Bmax - P) * invD
    t_enter = np.max(np.minimum(t0, t1), axis=2)
    t_exit  = np.min(np.maximum(t0, t1), axis=2)
    hit = (t_exit > t_enter) & (t_exit > 0)
    dists = np.where(hit, np.maximum(0.0, t_enter), np.inf)
    return np.clip(np.min(dists, axis=1), 0.0, max_range).astype(np.float32)

# clearance
def dist_point_to_aabb(px, py, cx, cy, hx, hy):
    dx = max(abs(px - cx) - hx, 0.0)
    dy = max(abs(py - cy) - hy, 0.0)
    return math.hypot(dx, dy)

def clearance_at_point(px, py, obs_boxes):
    best = 1e9
    for (cx, cy, hx, hy) in obs_boxes:
        best = min(best, dist_point_to_aabb(px, py, float(cx), float(cy), float(hx), float(hy)))
    dx = cone_xy[:, 0] - px
    dy = cone_xy[:, 1] - py
    center_dist = np.sqrt(dx*dx + dy*dy)
    best = min(best, float(np.min(center_dist - cone_xy[:, 2])))
    return float(best)

# bicycle
WHEELBASE = 2.8
def integrate_bicycle(px, py, yaw, v, delta, dt):
    yaw_rate = (v / WHEELBASE) * math.tan(delta)
    yaw = wrap_pi(yaw + yaw_rate * dt)
    px += v * math.sin(yaw) * dt
    py += v * math.cos(yaw) * dt
    return px, py, yaw, yaw_rate

# controller constants + functions (unchanged)
MAX_STEER = 0.55
BASE_SPEED = 8.0
MIN_SPEED = 2.0

def nominal_path_steer(px, py, yaw, s_est):
    Ld = 6.0 + 0.3 * BASE_SPEED
    gx, gy = get_oval_pos(s_est + Ld, 0.0)
    goal_yaw = math.atan2((gx - px), (gy - py))
    yaw_err = wrap_pi(goal_yaw - yaw)
    return float(np.clip(1.2 * yaw_err, -MAX_STEER, +MAX_STEER))

STEER_ENVELOPE = 0.45
N_CAND = 9
FORWARD_HORIZON = 10.0
EVAL_STEPS = 10
W_CLEAR = 1.0
W_DEV = 0.35
OBSTACLE_THRESH = 8.0
CRITICAL_THRESH = 4.5

def evaluate_candidate(px, py, yaw, v, delta_cand, delta_nom, s_est_guess, obs_boxes):
    dt_sim = FORWARD_HORIZON / EVAL_STEPS
    sim_px, sim_py, sim_yaw = px, py, yaw
    min_clear = 1e9
    cte_acc = 0.0
    s_est = s_est_guess
    for _ in range(EVAL_STEPS):
        sim_px, sim_py, sim_yaw, _ = integrate_bicycle(sim_px, sim_py, sim_yaw, v, delta_cand, dt_sim)
        min_clear = min(min_clear, clearance_at_point(sim_px, sim_py, obs_boxes))
        s_est, _, _, _, _, _, _, cte = project_to_centerline(sim_px, sim_py, s_est, samples=11)
        cte_acc += abs(cte)
    avg_cte = cte_acc / EVAL_STEPS
    score = (W_CLEAR * min_clear) - (W_DEV * abs(delta_cand - delta_nom)) - (0.08 * avg_cte)
    return float(score), float(min_clear)

def active_avoidance(px, py, yaw, v, delta_nom, s_est_guess, obs_boxes):
    cand = np.linspace(-STEER_ENVELOPE, +STEER_ENVELOPE, N_CAND)
    best_score = -1e9
    best_steer = float(delta_nom)
    best_clear = 1e9
    for d in cand:
        score, min_clear = evaluate_candidate(px, py, yaw, v, float(d), delta_nom, s_est_guess, obs_boxes)
        if score > best_score:
            best_score = score
            best_steer = float(d)
            best_clear = float(min_clear)
    return best_steer, best_clear

STEER_RATE_LIMIT = 2.0
STEER_DAMP = 0.75
MAX_LAT_ACCEL = 5.0
MIN_SPEED_FOR_STEER = 0.5

def anti_spin(delta_cmd, delta_prev, v, dt):
    max_step = STEER_RATE_LIMIT * dt
    delta_rl = float(np.clip(delta_cmd, delta_prev - max_step, delta_prev + max_step))
    v_eff = max(v, MIN_SPEED_FOR_STEER)
    tan_lim = (MAX_LAT_ACCEL * WHEELBASE) / (v_eff * v_eff)
    delta_dyn = math.atan(max(0.0, tan_lim))
    delta_dyn = min(delta_dyn, MAX_STEER)
    delta_mag = float(np.clip(delta_rl, -delta_dyn, +delta_dyn))
    delta_out = STEER_DAMP * delta_prev + (1.0 - STEER_DAMP) * delta_mag
    return float(delta_out), float(delta_dyn)

# obstacle sampling wrapper (unchanged control)
def sample_episode_obstacles(num_obs):
    fracs = []
    f = np.random.uniform(0.08, 0.14)
    for _ in range(num_obs):
        fracs.append(f % 1.0)
        f += np.random.uniform(0.10, 0.17)
    lat_mag_lo = 1.4
    lat_mag_hi = min(2.1, max(1.6, LANE_SAFE - 0.05))
    side = 1.0 if np.random.rand() > 0.5 else -1.0

    half = OB_SIZE / 2.0
    boxes = []
    for i in range(MAX_OBS):
        if i < num_obs:
            d = fracs[i] * perimeter
            lat = float(np.random.uniform(lat_mag_lo, lat_mag_hi) * side)
            side *= -1.0
            (_, _), (nx, ny) = get_tangent_normal(d)
            cx, cy = get_oval_pos(d, 0.0)
            ox = cx + lat * nx
            oy = cy + lat * ny
            obstacle_entities[i].set_pos((float(ox), float(oy), OB_SIZE / 2))
            obstacle_entities[i].set_quat((1.0, 0.0, 0.0, 0.0))
            boxes.append((float(ox), float(oy), float(half), float(half)))
        else:
            obstacle_entities[i].set_pos((0.0, 0.0, -100.0))
    return np.array(boxes, dtype=np.float32), num_obs

def run_episode():
    num_obs = int(np.random.randint(5, 8))
    obs_boxes, _ = sample_episode_obstacles(num_obs)

    px, py, pz, yaw, s_est = float(start_x), float(start_y - 5.0), 0.8, 0.0, 0.0
    delta = 0.0
    set_pose_yaw(car, px, py, pz, yaw)
    scene.step()

    steps = int(EPISODE_DURATION / sim_dt)

    lidar_list, cte_list, yaw_list = [], [], []
    speed_list, steer_list = [], []
    clear_list, coll_list, ts_list = [], [], []

    collided = False
    min_clear = 1e9

    for step in range(steps):
        s_est, _, _, _, _, _, _, cte = project_to_centerline(px, py, s_est, samples=15)

        ranges = lidar_scan_fast(px, py, yaw, ray_angles, obs_boxes, LIDAR_MAX)
        front_min = float(np.min(ranges[(np.abs(ray_angles) < math.radians(20.0))]))

        delta_nom = nominal_path_steer(px, py, yaw, s_est)

        if front_min < OBSTACLE_THRESH:
            delta_best, min_clear_best = active_avoidance(px, py, yaw, BASE_SPEED, delta_nom, s_est, obs_boxes)
            w = float(np.clip((OBSTACLE_THRESH - front_min) / max(OBSTACLE_THRESH - CRITICAL_THRESH, 1e-6), 0.0, 1.0))
            delta_cmd = (1.0 - w) * delta_nom + w * delta_best
        else:
            delta_cmd = delta_nom
            min_clear_best = 1e9

        speed = BASE_SPEED
        if front_min < CRITICAL_THRESH or min_clear_best < 1.8:
            speed = MIN_SPEED

        delta, _ = anti_spin(delta_cmd, delta, speed, sim_dt)
        delta = float(np.clip(delta, -MAX_STEER, +MAX_STEER))

        px, py, yaw, _ = integrate_bicycle(px, py, yaw, speed, delta, sim_dt)

        s_est, cx, cy, _, _, nx, ny, cte = project_to_centerline(px, py, s_est, window=18.0, samples=21)
        if abs(cte) > LANE_SAFE:
            cte_clip = float(np.clip(cte, -LANE_SAFE, +LANE_SAFE))
            px = float(cx + cte_clip * nx)
            py = float(cy + cte_clip * ny)
            delta *= 0.6

        set_pose_yaw(car, px, py, pz, yaw)
        set_joint_positions([delta, delta])
        scene.step()

        clearance = float(clearance_at_point(px, py, obs_boxes))
        min_clear = min(min_clear, clearance)
        is_collision = bool(clearance < 0.5)
        if is_collision:
            collided = True

        if MIN_CLEARANCE_FILTER > 0.0 and clearance < MIN_CLEARANCE_FILTER:
            if STOP_ON_COLLISION and is_collision:
                break
            continue

        lidar_list.append(ranges.astype(np.float32))
        cte_list.append(np.float32(cte))
        yaw_list.append(np.float32(yaw))
        speed_list.append(np.float32(speed))
        steer_list.append(np.float32(delta_cmd))
        clear_list.append(np.float32(clearance))
        coll_list.append(bool(is_collision))
        ts_list.append(np.float64(step * sim_dt))

        if STOP_ON_COLLISION and is_collision:
            break

    return {
        "observations/lidar_ranges": np.asarray(lidar_list, dtype=np.float32),
        "observations/cte": np.asarray(cte_list, dtype=np.float32),
        "yaws": np.asarray(yaw_list, dtype=np.float32),
        "velocities": np.asarray(speed_list, dtype=np.float32),
        "actions/steering": np.asarray(steer_list, dtype=np.float32),
        "metrics/clearances": np.asarray(clear_list, dtype=np.float32),
        "collisions": np.asarray(coll_list, dtype=bool),
        "timestamps": np.asarray(ts_list, dtype=np.float64),
        "min_clearance": float(min_clear),
        "collision_occurred": bool(collided),
        "num_steps": int(len(ts_list)),
        "num_obs": int(num_obs),
    }

def main():
    all_eps = []
    success = 0
    print("=" * 70)
    print("PHASE 2 FAST (STRICT)")
    print("=" * 70)

    for ep in range(NUM_EPISODES):
        epd = run_episode()
        ok = not epd["collision_occurred"]
        success += int(ok)
        if (ok or (not KEEP_ONLY_SUCCESS)) and epd["num_steps"] > 0:
            all_eps.append(epd)
        print(f"ep {ep:03d} | steps={epd['num_steps']:4d} | min_clear={epd['min_clearance']:.2f} | {'✅' if ok else '💥'} | kept={len(all_eps)}")

    with h5py.File(OUT_H5, "w") as f:
        f.attrs["sim_dt"] = sim_dt
        f.attrs["num_episodes_generated"] = NUM_EPISODES
        f.attrs["num_episodes_saved"] = len(all_eps)
        f.attrs["episode_duration"] = EPISODE_DURATION
        f.create_dataset("lidar_angles", data=ray_angles.astype(np.float32))
        for k, epd in enumerate(all_eps):
            g = f.create_group(f"episode_{k:03d}")
            for name in [
                "observations/lidar_ranges","observations/cte","yaws","velocities",
                "actions/steering","metrics/clearances","collisions","timestamps"
            ]:
                g.create_dataset(name, data=epd[name], compression="gzip", compression_opts=4 if name != "collisions" else None)
            g.attrs["num_steps"] = epd["num_steps"]
            g.attrs["collision_occurred"] = int(epd["collision_occurred"])
            g.attrs["min_clearance"] = float(epd["min_clearance"])
            g.attrs["num_obs"] = int(epd["num_obs"])

    print("\n✅ wrote:", OUT_H5)
    print(f"success among generated: {success}/{NUM_EPISODES} | saved: {len(all_eps)}")

if __name__ == "__main__":
    main()

[I 02/28/26 16:54:33.538 12180] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


LANE_SAFE = 3.2
PHASE 2 FAST (STRICT)
ep 000 | steps= 111 | min_clear=0.41 | 💥 | kept=0
ep 001 | steps= 300 | min_clear=1.26 | ✅ | kept=1
ep 002 | steps= 123 | min_clear=0.50 | 💥 | kept=1
ep 003 | steps= 300 | min_clear=0.88 | ✅ | kept=2
ep 004 | steps= 300 | min_clear=0.98 | ✅ | kept=3
ep 005 | steps= 300 | min_clear=0.85 | ✅ | kept=4
ep 006 | steps= 300 | min_clear=0.79 | ✅ | kept=5
ep 007 | steps= 117 | min_clear=0.40 | 💥 | kept=5
ep 008 | steps= 118 | min_clear=0.48 | 💥 | kept=5
ep 009 | steps= 300 | min_clear=1.12 | ✅ | kept=6
ep 010 | steps= 300 | min_clear=0.56 | ✅ | kept=7
ep 011 | steps= 300 | min_clear=0.65 | ✅ | kept=8
ep 012 | steps= 300 | min_clear=0.52 | ✅ | kept=9
ep 013 | steps= 114 | min_clear=0.47 | 💥 | kept=9
ep 014 | steps= 300 | min_clear=1.39 | ✅ | kept=10
ep 015 | steps= 116 | min_clear=0.49 | 💥 | kept=10
ep 016 | steps= 121 | min_clear=0.46 | 💥 | kept=10
ep 017 | steps= 300 | min_clear=1.50 | ✅ | kept=11
ep 018 | steps= 100 | min_clear=0.41 | 💥 | kept=11
ep 019 